# 03.1 Data Preparation

Вход: `european_flights_annotated_v3.parquet` (149M строк, 41 колонка).
Выход:
- `split_metadata.json`: flight_id assignments train_core / calibration_val / test.
- `clip_bounds.json`: P0.1/P99.9 границы каждой из 12 model features.
- `scaler.joblib`: StandardScaler, fit на clipped clean train_core.

## Что делает 03.1

Готовит данные для всех трёх моделей (IF, HDBSCAN, LSTM-AE) одной согласованной процедурой:

1. Train/test split по `flight_start_time`: train_core (80% train), calibration_val (20% train), test (после TRAIN_END). Никогда не режем рейс между splits.
2. DQ-aware eligibility masks:
   - `clean_train_mask`: для training всех моделей.
   - `score_eligible_mask`: для scoring (всё с finite features).
3. Train-only clipping P0.1/P99.9 на clean train_core sample.
4. StandardScaler fit только на clipped clean train_core.
5. Сохранение артефактов для воспроизводимости 03.2-03.5.

## Контракт с downstream секциями

После 03.1 все остальные секции:
- Читают `annotated_v3.parquet` и `split_metadata.json` для определения своих splits.
- Применяют clip с последующим scaler из артефактов (никогда не пересчитывают).
- Используют один и тот же `clean_train_mask` для training pool.
- Test трогают только в финальной evaluation (03.5).

## Что 03.1 НЕ делает

- Не загружает весь датасет в память (149M строк x 41 колонка > 25 GB).
- Не модифицирует annotated_v3.
- Не считает scores.
- Не строит модели.

Все операции на streaming chunks через pyarrow плюс накопление light-weight аггрегатов (per-flight first timestamp, clip quantiles на sample).

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import os
import gc
import json
import time
from sklearn.preprocessing import StandardScaler
import joblib

DATA_DIR = '/content/drive/MyDrive/thesis_processed'
ARTIFACTS_DIR = os.path.join(DATA_DIR, 'models_v3_artifacts')
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

INPUT_PATH = os.path.join(DATA_DIR, 'european_flights_annotated_v3.parquet')

# Output artifacts
SPLIT_PATH = os.path.join(ARTIFACTS_DIR, 'split_metadata.json')
CLIP_BOUNDS_PATH = os.path.join(ARTIFACTS_DIR, 'clip_bounds.json')
SCALER_PATH = os.path.join(ARTIFACTS_DIR, 'scaler.joblib')

print(f'Input:  {INPUT_PATH}')
print(f'Artifacts: {ARTIFACTS_DIR}')
print(f'Input exists: {os.path.exists(INPUT_PATH)}')
print(f'Input size: {os.path.getsize(INPUT_PATH) / 1e9:.2f} GB')

Mounted at /content/drive
Input:  /content/drive/MyDrive/thesis_processed/european_flights_annotated_v3.parquet
Artifacts: /content/drive/MyDrive/thesis_processed/models_v3_artifacts
Input exists: True
Input size: 8.36 GB


In [2]:
# Константы
# 12 model features из контракта 02b_v4
MODEL_FEATURES = [
    'altitude', 'groundspeed', 'vertical_rate',
    'acceleration', 'turn_rate', 'vertical_accel',
    'wind_speed', 'headwind', 'crosswind',
    'energy_ratio', 'energy_rate', 'energy_deviation',
]

# DQ-флаги из контракта 02c_v3
DQ_HARD = 'dq_hard_flag'
DQ_SOFT = 'dq_soft_flag'
FEATURE_QUALITY = 'feature_quality_flag'
GAP_FLAG = 'gap_flag'

# Параметры split
TRAIN_END = pd.Timestamp('2022-11-10 23:59:59', tz='UTC')
CALIBRATION_FRACTION = 0.20  # 20% train flights в calibration_val

# Перцентили для clipping
CLIP_Q_LOW = 0.001    # P0.1
CLIP_Q_HIGH = 0.999   # P99.9

# Sampling для оценки clip bounds
# Берём sample из clean train_core, не весь pool (иначе RAM)
CLIP_SAMPLE_SIZE = 5_000_000  # 5M точек достаточно для робастных квантилей

# Воспроизводимость
RANDOM_STATE = 1321

print(f'\nMODEL_FEATURES ({len(MODEL_FEATURES)}):')
for f in MODEL_FEATURES:
    print(f'  {f}')
print(f'\nTRAIN_END: {TRAIN_END}')
print(f'Calibration fraction: {CALIBRATION_FRACTION:.0%}')
print(f'Clip percentiles: P{CLIP_Q_LOW * 100:.1f} / P{CLIP_Q_HIGH * 100:.1f}')
print(f'Clip sample size: {CLIP_SAMPLE_SIZE:,}')


MODEL_FEATURES (12):
  altitude
  groundspeed
  vertical_rate
  acceleration
  turn_rate
  vertical_accel
  wind_speed
  headwind
  crosswind
  energy_ratio
  energy_rate
  energy_deviation

TRAIN_END: 2022-11-10 23:59:59+00:00
Calibration fraction: 20%
Clip percentiles: P0.1 / P99.9
Clip sample size: 5,000,000


## 2. Train/test split via flight_start_time

Один проход через parquet, читаем только flight_id и timestamp. После этого решаем, какие flights принадлежат:
- test: первый timestamp > TRAIN_END.
- train_core: 80% train flights (random by flight_id).
- calibration_val: 20% train flights (random by flight_id).

Calibration split нужен для:
- IF: percentile normalization scores (если бы делали на train_core, был бы перекос, модель видела эти точки).
- HDBSCAN: то же.
- LSTM-AE: validation loss для early stopping плюс score normalization.
- 03.5: calibration of risk thresholds.

Test трогается только в финальной evaluation. Никаких threshold/scaling из test.

In [3]:
print('Computing flight_start_time and split assignments...')
t0 = time.time()

# Читаем только нужные колонки
ts_table = pq.read_table(INPUT_PATH, columns=['flight_id', 'timestamp']).to_pandas()
ts_table['timestamp'] = pd.to_datetime(ts_table['timestamp'])

flight_starts = ts_table.groupby('flight_id')['timestamp'].min()
n_total = len(flight_starts)

# Test: рейсы с первой точкой > TRAIN_END
test_flights_mask = flight_starts > TRAIN_END
test_flights = flight_starts[test_flights_mask].index.to_numpy()

# Train pool: всё остальное
train_pool = flight_starts[~test_flights_mask].index.to_numpy()
n_train_pool = len(train_pool)

# Внутри train pool: 80% train_core / 20% calibration_val (random by flight_id)
rng = np.random.RandomState(RANDOM_STATE)
shuffled = rng.permutation(train_pool)
n_calibration = int(n_train_pool * CALIBRATION_FRACTION)
calibration_val_flights = shuffled[:n_calibration]
train_core_flights = shuffled[n_calibration:]

print(f'\nSplit summary:')
print(f'  Total flights:        {n_total:,}')
print(f'  Train pool (≤ TRAIN_END):  {n_train_pool:,}')
print(f'    train_core (80%):     {len(train_core_flights):,}')
print(f'    calibration_val (20%): {len(calibration_val_flights):,}')
print(f'  Test (> TRAIN_END):   {len(test_flights):,}')

del ts_table, flight_starts

print(f'\nSplit computation: {time.time() - t0:.0f}s')

Computing flight_start_time and split assignments...

Split summary:
  Total flights:        29,788
  Train pool (≤ TRAIN_END):  21,537
    train_core (80%):     17,230
    calibration_val (20%): 4,307
  Test (> TRAIN_END):   8,251

Split computation: 74s


In [4]:
# Sanity: проверяем, что splits не пересекаются
train_core_set = set(train_core_flights.tolist())
calibration_val_set = set(calibration_val_flights.tolist())
test_set = set(test_flights.tolist())

assert len(train_core_set & calibration_val_set) == 0
assert len(train_core_set & test_set) == 0
assert len(calibration_val_set & test_set) == 0
assert len(train_core_set | calibration_val_set | test_set) == n_total
print('Splits are mutually exclusive and exhaustive.')

# Сохраняем split metadata
split_metadata = {
    'train_end_utc': TRAIN_END.isoformat(),
    'calibration_fraction': CALIBRATION_FRACTION,
    'random_state': RANDOM_STATE,
    'n_total': int(n_total),
    'n_train_core': len(train_core_flights),
    'n_calibration_val': len(calibration_val_flights),
    'n_test': len(test_flights),
    'train_core_flights': train_core_flights.tolist(),
    'calibration_val_flights': calibration_val_flights.tolist(),
    'test_flights': test_flights.tolist(),
}

with open(SPLIT_PATH, 'w') as f:
    json.dump(split_metadata, f)
print(f'Saved: {SPLIT_PATH} ({os.path.getsize(SPLIT_PATH) / 1e6:.2f} MB)')

Splits are mutually exclusive and exhaustive. ✓
Saved: /content/drive/MyDrive/thesis_processed/models_v3_artifacts/split_metadata.json (0.33 MB)


## 3. Discover row counts per split

Прогон по annotated_v3, чтобы посчитать row counts и DQ-clean rates per split. Для логов и sanity check, что splits разумные.

Также строим chunk plan: разбиваем train_core_flights, calibration_val, test на chunks для дальнейших streaming операций (clip sample, scaler fit).

In [5]:
print('Counting rows per split + computing clean fractions...')
t0 = time.time()

# Reload split sets (после json потеряли тип int если было int64)
train_core_set = set(int(f) for f in train_core_flights)
calibration_val_set = set(int(f) for f in calibration_val_flights)
test_set = set(int(f) for f in test_flights)

# Streaming чтение minimum columns: flight_id + DQ flags
# (для row count и clean check)
min_cols = ['flight_id', DQ_HARD, DQ_SOFT, FEATURE_QUALITY] + MODEL_FEATURES

# Чанковое чтение через pq.ParquetFile.iter_batches
pf = pq.ParquetFile(INPUT_PATH)
total_rows = pf.metadata.num_rows
print(f'Total rows: {total_rows:,}')

# Накапливаем counts
counts = {
    'train_core': {'total': 0, 'clean': 0, 'finite': 0},
    'calibration_val': {'total': 0, 'clean': 0, 'finite': 0},
    'test': {'total': 0, 'clean': 0, 'finite': 0},
}

batch_size = 5_000_000  # 5M строк на batch
n_batches_processed = 0

for batch in pf.iter_batches(batch_size=batch_size, columns=min_cols):
    df_batch = batch.to_pandas()

    # Determine split for each row
    fid = df_batch['flight_id'].to_numpy()
    is_train_core = np.isin(fid, list(train_core_set))
    is_calibration_val = np.isin(fid, list(calibration_val_set))
    is_test = np.isin(fid, list(test_set))

    # finite_mask: все 12 model features finite
    finite_mask = df_batch[MODEL_FEATURES].notna().all(axis=1).to_numpy()

    # clean_mask: finite + no DQ flags
    clean_mask = (
        finite_mask
        & (df_batch[DQ_HARD].to_numpy() == 0)
        & (df_batch[DQ_SOFT].to_numpy() == 0)
        & (df_batch[FEATURE_QUALITY].to_numpy() == 0)
    )

    for split_name, split_mask in [
        ('train_core', is_train_core),
        ('calibration_val', is_calibration_val),
        ('test', is_test),
    ]:
        counts[split_name]['total'] += int(split_mask.sum())
        counts[split_name]['finite'] += int((split_mask & finite_mask).sum())
        counts[split_name]['clean'] += int((split_mask & clean_mask).sum())

    n_batches_processed += 1
    print(f'  batch {n_batches_processed}: {len(df_batch):,} rows', end='\r')

    del df_batch, fid, is_train_core, is_calibration_val, is_test
    del finite_mask, clean_mask

print()
print(f'\nRow counts per split:')
print(f'{"split":>20s} | {"total":>12s} | {"finite":>12s} | {"clean":>12s} | {"clean%":>7s}')
print('-' * 80)
total_check = 0
for split_name, c in counts.items():
    pct = c['clean'] / c['total'] * 100 if c['total'] > 0 else 0
    print(f'{split_name:>20s} | {c["total"]:,} | {c["finite"]:,} | '
          f'{c["clean"]:,} | {pct:>6.2f}%')
    total_check += c['total']

print('-' * 80)
print(f'  Total check:       {total_check:,} (expected {total_rows:,})')
assert total_check == total_rows
print('\nRow count consistency: OK')

print(f'\nCounts pass: {time.time() - t0:.0f}s')

Counting rows per split + computing clean fractions...
Total rows: 149,129,454
  batch 30: 4,129,454 rows

Row counts per split:
               split |        total |       finite |        clean |  clean%
--------------------------------------------------------------------------------
          train_core |   86,109,651 |   85,556,403 |   84,382,917 |  97.99%
     calibration_val |   21,526,151 |   21,390,254 |   21,083,531 |  97.94%
                test |   41,493,652 |   41,188,507 |   40,611,544 |  97.87%
--------------------------------------------------------------------------------
  Total check:       149,129,454 (expected 149,129,454)

Row count consistency: ✓

Counts pass: 43s


## 4. Sample for clip bounds estimation

Для расчёта P0.1/P99.9 на clean train_core берём random sample размером CLIP_SAMPLE_SIZE точек. На 5M точках квантили уже robust (SE < 0.01% для extreme percentiles).

Загружаем весь sample в RAM, нужны exact quantiles. 5M x 12 features x float64 ≈ 480 МБ, спокойно влезает.

In [6]:
clean_train_total = counts['train_core']['clean']
print(f'Clean train_core size: {clean_train_total:,}')
print(f'Sampling {CLIP_SAMPLE_SIZE:,} points for clip bounds...')
t0 = time.time()

# Sampling rate: вероятность взять каждую clean train_core точку
sample_rate = min(CLIP_SAMPLE_SIZE / clean_train_total, 1.0)
print(f'Sample rate: {sample_rate:.2f}')

# Stream через batches, decide per-row
rng = np.random.RandomState(RANDOM_STATE + 1)  # отдельный seed для sample
sample_chunks = []

for batch in pf.iter_batches(batch_size=batch_size, columns=min_cols):
    df_batch = batch.to_pandas()

    fid = df_batch['flight_id'].to_numpy()
    is_train_core = np.isin(fid, list(train_core_set))

    finite_mask = df_batch[MODEL_FEATURES].notna().all(axis=1).to_numpy()

    clean_mask = (
        finite_mask
        & is_train_core
        & (df_batch[DQ_HARD].to_numpy() == 0)
        & (df_batch[DQ_SOFT].to_numpy() == 0)
        & (df_batch[FEATURE_QUALITY].to_numpy() == 0)
    )

    if clean_mask.sum() == 0:
        del df_batch, fid, is_train_core, finite_mask, clean_mask
        continue

    # Применяем sample rate
    random_pick = rng.random(len(df_batch)) < sample_rate
    selected = clean_mask & random_pick

    if selected.sum() > 0:
        sample_chunks.append(df_batch.loc[selected, MODEL_FEATURES].to_numpy())

    del df_batch, fid, is_train_core, finite_mask, clean_mask, random_pick, selected

clip_sample = np.vstack(sample_chunks).astype(np.float32)
del sample_chunks

print(f'Sample collected: {len(clip_sample):,} rows × {clip_sample.shape[1]} features')
print(f'Sample memory: {clip_sample.nbytes / 1e6:.1f} MB')
print(f'Sampling time: {time.time() - t0:.0f}s')

Clean train_core size: 84,382,917
Sampling 5,000,000 points for clip bounds...
Sample rate: 0.0593
Sample collected: 4,999,795 rows × 12 features
Sample memory: 240.0 MB
Sampling time: 26s


## 5. Compute clip bounds (P0.1 / P99.9 per feature)

Для каждой из 12 model features считаем `[CLIP_Q_LOW, CLIP_Q_HIGH]` квантили на clean train_core sample. Эти границы используются для clipping всех feature columns (train, calibration_val, test) одинаково.

In [7]:
print('Computing clip bounds...')

clip_bounds = {}
print(f'\n{"feature":>20s} | {"P0.1":>14s} | {"P99.9":>14s} | {"width":>14s}')
print('-' * 75)

for i, feature in enumerate(MODEL_FEATURES):
    vals = clip_sample[:, i]
    q_low = float(np.quantile(vals, CLIP_Q_LOW))
    q_high = float(np.quantile(vals, CLIP_Q_HIGH))
    clip_bounds[feature] = {'low': q_low, 'high': q_high}
    print(f'{feature:>20s} | {q_low:>14.4f} | {q_high:>14.4f} | {q_high - q_low:>14.4f}')

# Save
with open(CLIP_BOUNDS_PATH, 'w') as f:
    json.dump({
        'percentile_low': CLIP_Q_LOW,
        'percentile_high': CLIP_Q_HIGH,
        'sample_size': len(clip_sample),
        'random_state': RANDOM_STATE + 1,
        'bounds': clip_bounds,
    }, f, indent=2)
print(f'\nSaved: {CLIP_BOUNDS_PATH}')

# Sanity: показать долю sample за bounds (должно быть ~0.2% по дизайну
# из-за P0.1/P99.9 - снизу и сверху по 0.1%)
print(f'\nFraction outside bounds (per feature, sanity check ~0.2%):')
for i, feature in enumerate(MODEL_FEATURES):
    vals = clip_sample[:, i]
    bounds = clip_bounds[feature]
    n_out = int(((vals < bounds['low']) | (vals > bounds['high'])).sum())
    pct = n_out / len(vals) * 100
    print(f'  {feature:>20s}: {n_out:,} ({pct:>5.3f}%)')

Computing clip bounds...

             feature |           P0.1 |          P99.9 |          width
---------------------------------------------------------------------------
            altitude |         0.0000 |     41000.0000 |     41000.0000
         groundspeed |        94.0000 |       542.0000 |       448.0000
       vertical_rate |     -3840.0000 |      4288.0000 |      8128.0000
        acceleration |        -1.9000 |         2.3000 |         4.2000
           turn_rate |        -2.5934 |         2.4884 |         5.0818
      vertical_accel |      -166.4000 |       172.8000 |       339.2000
          wind_speed |         0.5717 |        64.1558 |        63.5841
            headwind |       -47.4719 |        47.9827 |        95.4547
           crosswind |         0.0166 |        53.9364 |        53.9198
        energy_ratio |         0.0130 |         1.0000 |         0.9870
         energy_rate |       -29.4052 |        30.4801 |        59.8853
    energy_deviation |        -4.4

## 6. Fit StandardScaler on clipped clean train_core sample

Применяем clip bounds к sample, fit StandardScaler. Сохраняем для downstream. После этого model features считаются в scaled space:

```
X_scaled = scaler.transform(np.clip(X_raw, low, high))
```

Это применяется к calibration_val, test, и при scoring.

In [8]:
print('Clipping sample and fitting StandardScaler...')

# Apply clip
clip_sample_clipped = clip_sample.copy()
for i, feature in enumerate(MODEL_FEATURES):
    bounds = clip_bounds[feature]
    np.clip(clip_sample_clipped[:, i], bounds['low'], bounds['high'],
            out=clip_sample_clipped[:, i])

# Fit scaler
scaler = StandardScaler()
scaler.fit(clip_sample_clipped)

# Save
joblib.dump(scaler, SCALER_PATH)
print(f'Saved: {SCALER_PATH}')

# Show resulting scaler params
print(f'\nScaler params (after clip):')
print(f'{"feature":>20s} | {"mean":>14s} | {"std":>14s}')
print('-' * 60)
for i, feature in enumerate(MODEL_FEATURES):
    print(f'{feature:>20s} | {scaler.mean_[i]:>14.4f} | {scaler.scale_[i]:>14.4f}')

# Sanity: scaled sample должен иметь mean ≈ 0, std ≈ 1
scaled_sample = scaler.transform(clip_sample_clipped)
print(f'\nScaled sample mean (should ≈ 0): {scaled_sample.mean(axis=0).round(6)}')
print(f'Scaled sample std (should ≈ 1):  {scaled_sample.std(axis=0).round(6)}')

del clip_sample, clip_sample_clipped, scaled_sample

Clipping sample and fitting StandardScaler...
Saved: /content/drive/MyDrive/thesis_processed/models_v3_artifacts/scaler.joblib

Scaler params (after clip):
             feature |           mean |            std
------------------------------------------------------------
            altitude |     26311.6003 |     12529.2242
         groundspeed |       381.1519 |        98.5584
       vertical_rate |       -21.2236 |      1141.8403
        acceleration |        -0.0026 |         0.3021
           turn_rate |        -0.0018 |         0.2929
      vertical_accel |        -0.4541 |        24.0919
          wind_speed |        19.0354 |        10.6061
            headwind |         0.9155 |        15.0228
           crosswind |        12.1568 |         9.9483
        energy_ratio |         0.2338 |         0.0873
         energy_rate |        -0.0107 |         7.1412
    energy_deviation |         0.0085 |         0.7590

Scaled sample mean (should ≈ 0): [ 2.0e-05  2.0e-06 -1.3e-05  3.0e-

0

## 7. Сохраняем MODEL_FEATURES и DQ_FLAGS контракт

Дополнительный артефакт: список 12 model features в порядке (для reproducibility), список DQ flags. 03.2-03.5 импортируют эти константы.

In [9]:
contract = {
    'model_features': MODEL_FEATURES,
    'dq_hard_flag': DQ_HARD,
    'dq_soft_flag': DQ_SOFT,
    'feature_quality_flag': FEATURE_QUALITY,
    'gap_flag': GAP_FLAG,
    'train_end_utc': TRAIN_END.isoformat(),
    'calibration_fraction': CALIBRATION_FRACTION,
    'clip_q_low': CLIP_Q_LOW,
    'clip_q_high': CLIP_Q_HIGH,
    'random_state': RANDOM_STATE,
}
contract_path = os.path.join(ARTIFACTS_DIR, 'contract.json')
with open(contract_path, 'w') as f:
    json.dump(contract, f, indent=2)
print(f'Saved: {contract_path}')

Saved: /content/drive/MyDrive/thesis_processed/models_v3_artifacts/contract.json


## 8. Summary

03.1 завершён. Готовы три ключевых артефакта:

- `split_metadata.json`: flight_id и split (train_core / calibration_val / test).
- `clip_bounds.json`: P0.1/P99.9 границы 12 model features.
- `scaler.joblib`: StandardScaler, fit на clipped clean train_core.

Эти артефакты используются всеми последующими секциями (03.2-03.5) без модификации. Контракт зафиксирован.

### Чек-лист артефактов

In [10]:
print('\n' + '=' * 60)
print('03.1 SUMMARY')
print('=' * 60)
print(f'Splits:')
print(f'  train_core:      {len(train_core_flights):,} flights')
print(f'  calibration_val: {len(calibration_val_flights):,} flights')
print(f'  test:            {len(test_flights):,} flights')

print(f'\nClean train_core for IF/HDBSCAN/LSTM training: '
      f'{counts["train_core"]["clean"]:,} points')
print(f'  ({counts["train_core"]["clean"] / counts["train_core"]["total"] * 100:.1f}% of train_core)')

print(f'\nArtifacts:')
for path in [SPLIT_PATH, CLIP_BOUNDS_PATH, SCALER_PATH, contract_path]:
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1e3
        print(f'  OK   {os.path.basename(path):<25s} ({size_kb:>8.1f} KB)')
    else:
        print(f'  MISSING: {os.path.basename(path)}')

print('\n03.1 complete. Ready for 03.2 (Isolation Forest).')


03.1 SUMMARY
Splits:
  train_core:      17,230 flights
  calibration_val:  4,307 flights
  test:             8,251 flights

Clean train_core for IF/HDBSCAN/LSTM training: 84,382,917 points
  (98.0% of train_core)

Artifacts:
  ✓ split_metadata.json       (   327.9 KB)
  ✓ clip_bounds.json          (     1.2 KB)
  ✓ scaler.joblib             (     0.9 KB)
  ✓ contract.json             (     0.6 KB)

03.1 complete. Ready for 03.2 (Isolation Forest).
